# EDA — Knee MRI dataset

This notebook scans a root DICOM directory and summarizes studies, series, slice counts, and SeriesDescription keywords. Run on the raw DICOM folder before preprocessing.

In [ ]:
import os
from pathlib import Path
import pydicom
import pandas as pd
import matplotlib.pyplot as plt

# Set this to your DICOM root folder
DICOM_ROOT = '/path/to/dicom_root'
root = Path(DICOM_ROOT)
assert root.exists(), 'Set DICOM_ROOT to the directory with DICOM files'

records = []
for p in root.rglob('*'):
    if p.is_file() and p.suffix.lower() in ('.dcm', ''):
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True)
            records.append({
                'path': str(p),
                'study_uid': getattr(ds, 'StudyInstanceUID', None),
                'series_uid': getattr(ds, 'SeriesInstanceUID', None),
                'series_desc': getattr(ds, 'SeriesDescription', ''),
            })
        except Exception:
            continue

df = pd.DataFrame(records)
print('Found files:', len(df))

# drop rows without study/series
df = df.dropna(subset=['study_uid', 'series_uid'])

# aggregate
agg = df.groupby(['study_uid', 'series_uid'])['path'].count().reset_index().rename(columns={'path': 'n_files'})
# bring series_desc
desc = df.groupby(['study_uid', 'series_uid'])['series_desc'].first().reset_index()
agg = agg.merge(desc, on=['study_uid', 'series_uid'])

print('Studies:', agg['study_uid'].nunique())
print('Total series:', agg.shape[0])

# top series descriptions
top_desc = agg['series_desc'].value_counts().head(30)
top_desc.plot(kind='bar', figsize=(12,6))
plt.title('Top SeriesDescription values')
plt.show()

# distribution of slices per series
agg['n_files'].hist(bins=50, figsize=(8,4))
plt.title('Slice count distribution per series')
plt.show()

# Save summary to CSV
agg.to_csv('eda_series_summary.csv', index=False)
agg.head()
